# ⚡ 13 · 预测式专家预取（Predictive Expert Prefetch）

> AI-Infra 面试高频：**MoE 的专家是按 token 动态路由的，命中与否事先不知道——那能不能"猜"它下一步要哪个专家，提前把权重搬进显存？**

- MoE 推理的单 token 只激活 `top-k` 个专家（如 $k=2$，共 $E=64$），但**每次激活都可能是冷专家** → 显存里装不下全部专家，只能按需换入 → 暴露 PCIe/NVLink 装载时延；
- 预测式预取的做法：用**上一层已经算出来的信息**（路由分数、已激活专家、hidden-state）猜下一层要谁，**在后台 DMA 里先把权重搬过来**，把串行 stall 变成可重叠的搬运；
- 但它有代价：**猜错就白占带宽、白挤缓存**。所以工程上真正要优化的是 **命中率 vs 浪费率** 的权衡，而不是单纯"命中率越高越好"。

本节用纯 numpy 手写三种预取器（上层基线 / 马尔可夫 / logistic），把它们接进一个带 LRU 专家缓存的模拟器，量出端到端加速比。

## 📋 本节要解决的问题

| # | 问题 | 答案 |
|---|---|---|
| 1 | 为什么不能把专家全放显存？ | $E$ 个专家 × 每层，权重总量远超显存；只能按需换入 |
| 2 | 不预取会怎样？ | 每次冷专家激活都串行等一次装载 → stall |
| 3 | 凭什么能预取？ | 路由分数由 hidden-state 决定，**层之间有时间相关**（上一层算完才知道下一层） |
| 4 | 最简单的预测器？ | 上层预取：直接复用上一层 top-k 下标 |
| 5 | 更强的是什么？ | 马尔可夫转移 $P(e_{next}\mid e_{cur})$；logistic/MLP 用 hidden-state 特征 |
| 6 | 预测错了怎么办？ | 白占带宽 + 挤掉缓存里的热专家 → 用**浪费率**量化这种伤害 |
| 7 | 怎么算"值不值"？ | 有效加速比 $S=\dfrac{\text{stall}_{base}}{\text{stall}_{pf}+\text{overhead}}$ |
| 8 | 什么时候会预测失败？ | 路由弱相关 / 负载均衡抖动 / 跨域 batch 混合 |

## 1️⃣ 直觉与大白话

### 类比 A：外卖预点单

你每天 12:05 点同一家店的宫保鸡丁。老板**提前 10 分钟**就把它炒好放在保温台——你一到就端走（**命中**）。
但如果今天你改吃别的，那盘宫保鸡丁就只能倒掉（**预取了但没用 = 浪费**）；更糟的是它**占着灶台**，让别人点的菜变慢（**挤占缓存/带宽**）。

预取器就是那个"猜你吃什么"的老板。猜得准 → 出餐快；猜不准 → 后厨更堵。

### 类比 B：地铁提前调度空车

早高峰前，调度中心把**空车提前开到大客流站**待命。乘客上车即走（命中），省掉了从车辆段调车的时间（stall）。
但如果客流预测错了，空车白跑一趟（浪费运力），而且它占着轨道，反而让别的车晚点（**带宽竞争**）。

### 映射回 MoE

| 现实角色 | 在本节模拟里 |
|---|---|
| 乘客 | token |
| 站点 | 层 $l$ |
| 菜品 / 空车 | 专家权重（`expert_mb` 8 MB） |
| 老板的预测 / 调度中心 | 预取器（上层 / 马尔可夫 / logistic） |
| 灶台 / 轨道 | 专家缓存槽位 + DMA 带宽 |
| 出餐等待 / 调车时间 | stall（`load_us` 160 µs） |
| 倒掉的菜 / 空跑的车 | 浪费率（预取未被使用） |

**核心张力**：预取的本质是"用带宽换时延"。带宽不是免费的，所以必须同时看**命中率**和**浪费率**。

## 2️⃣ 为什么必须"预测"（路由是数据相关的）

MoE 层的前向是：

$$
g(x)=\mathrm{softmax}(W_r x),\qquad \mathcal{E}(x)=\mathrm{TopK}\big(g(x)\big),\qquad
y=\sum_{e\in\mathcal{E}(x)} g_e(x)\,E_e(x)
$$

关键点：

1. **$\mathcal{E}(x)$ 依赖 $x$**（hidden-state），不是一个固定集合 → 无法静态预排权重；
2. **路由有强时间局部性**：相邻层的 hidden-state 高度相关（残差流 + 同一 token 语义不变），所以 $l-1$ 层激活的专家，对 $l$ 层是**很强的先验**；
3. **但只有弱相关时就没用**：路由若被负载均衡噪声 / 跨域 batch 打散，$l-1$ 的信息几乎不携带 $l$ 的信息 → 上层预取退化成随机猜。

这三条正好给出本节的两组分实验：**强层间相关**（$\rho=0.92$）vs **弱层间相关**（$\rho=0.10$），其中 $\rho$ 是合成 hidden-state 的 AR(1) 系数：

$$
h_l=\rho\,h_{l-1}+\sqrt{1-\rho^{2}}\;\varepsilon_l .
$$

$\rho\to 1$ ⟹ 相邻层 hidden 几乎不变 ⟹ ��由几乎不变 ⟹ 预取最容易；$\rho\to 0$ ⟹ 两层路由近似独立 ⟹ 只能靠全局热度兜底。

> 注意因果边界：**预取只能用在 $l-k$ 层已经算完的信息**去猜第 $l$ 层，绝不能用第 $l$ 层自己的路由结果——那是泄漏。

In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt

SEED = 20261120
np.random.seed(SEED)
t_start = time.time()

CFG = dict(
    n_layers=6,        # 层数 L
    n_experts=64,      # 每层专家数 E
    top_k=2,           # 每 token 每层激活数 k
    n_tokens=2048,     # 合成 token 数（= 一个 batch）
    chunk=8,           # microbatch 大小（打波/DMA 的最小调度单位）
    hidden=32,         # 合成 hidden-state 维度
    proj=10,           # logistic 用的随机投影维度
    noise=0.40,        # 路由噪声（负载均衡抖动强度）
    rho_strong=0.92,   # 强层间相关
    rho_weak=0.10,     # 弱层间相关
    expert_mb=8.0,     # 单个专家权重（MB）
    cache_slots=16,    # 专家缓存槽位数
    load_us=160.0,     # 未命中的暴露装载时延（µs）
    comp_us=18.0,      # 单专家单次计算时延（µs）
    dma_gbs=48.0,      # 预取 DMA 有效带宽（GB/s）
    overlap=0.85,      # DMA 与计算的重叠比例
    pf_width=4,        # 每次预测输出的专家数 m
    ntrain=1800,       # 预测器训练 token 数（必须 < n_tokens）
    epochs=30, patience=5, l2=4e-4, lr=0.12, bs=1024,
)
L, E, K, T = CFG['n_layers'], CFG['n_experts'], CFG['top_k'], CFG['n_tokens']
G, C = CFG['chunk'], T // CFG['chunk']
NTRAIN = CFG['ntrain']

pf_full_us = CFG['expert_mb'] * 1000.0 / CFG['dma_gbs']          # 单专家完整 DMA 时延
pf_cost_us = pf_full_us * (1.0 - CFG['overlap'])                # 只算未被重叠掩盖的部分
cache_mb = CFG['cache_slots'] * CFG['expert_mb']

print('层数 L=%d，每层专家 E=%d，top-k=%d，token=%d' % (L, E, K, T))
print('microbatch = %d token → 一个 batch 切成 %d 个 microbatch；逐层执行共 %d 个波次' % (G, C, L * C))
print('专家缓存：%d 槽 × %.0f MB = %.0f MB（全量专家需 %.0f MB，装不下 → 必须换入）'
      % (CFG['cache_slots'], CFG['expert_mb'], cache_mb, L * E * CFG['expert_mb']))
print('单专家完整 DMA ≈ %.1f µs，其中暴露给关键路径 ≈ %.1f µs（重叠 %.0f%%）'
      % (pf_full_us, pf_cost_us, CFG['overlap'] * 100))
print('未命中装载 stall = %.0f µs / 次' % CFG['load_us'])

## 3️⃣ 机制图

![预测式专家预取时间线](images/13_prefetch_timeline.svg)

读图三条线：

1. **Router 输出**（第 1 行）：第 $l$ 层算出 `top-k` 专家集合，此刻才能确定到底要谁；
2. **预测 + 后台 DMA**（第 2、3 行）：在第 $l$ 层的计算**之前**（即第 $l-1$ 层结束时）就发出预测，DMA 与前一层计算**并行**搬运；
3. **计算**（第 4 行）：
   - 命中（绿）：权重已在缓存，直接算，无 stall；
   - 未命中（红）：串行等一次 `load_us` —— 这就是预取要消灭的 stall；
   - **预取了但没用**（橙色斜纹）：不 stall，但占用了 DMA 带宽与缓存槽位 —— 浪费区间。

> 图中的橙色区间就是本节要量化的"浪费率"：它不改 stall，却真实消耗带宽，挤掉本可以命中的热专家。

In [ ]:
def make_trace(rho, seed):
    '''合成一条路由轨迹：(层, token) -> top-k 专家下标 + hidden-state。

    rho 控制相邻层 hidden 的 AR(1) 相关：rho→1 强相关，rho→0 近似独立。
    logits 做了跨 token 的均值归零，抵消"某些专家全局更热"的偏置，
    这样强/弱两组的差异只来自层间相关本身。
    '''
    rng = np.random.default_rng(seed)
    W = rng.normal(0.0, 1.0 / np.sqrt(CFG['hidden']), size=(E, CFG['hidden']))
    b = rng.normal(0.0, 0.15, size=E)

    hs = [rng.normal(0.0, 1.0, size=(T, CFG['hidden']))]
    for _ in range(1, L):
        hs.append(rho * hs[-1] + np.sqrt(max(1e-12, 1.0 - rho ** 2)) * rng.normal(0.0, 1.0, size=(T, CFG['hidden'])))
    hid = np.stack(hs, axis=0)                                   # L,T,H

    logits = hid @ W.T + b + rng.normal(0.0, CFG['noise'], size=(L, T, E))
    logits -= logits.mean(axis=1, keepdims=True)                  # 抵消全局热度偏置
    experts = np.argsort(-logits, axis=-1)[:, :, :K]              # L,T,K
    return dict(rho=rho, hid=hid, logits=logits, experts=experts)


def same_token_overlap(tr):
    '''同一 token 在相邻层激活的专家重合率（= 上层预取的"理论上限直觉"）'''
    ex = tr['experts']
    hit = 0
    for l in range(1, L):
        for t in range(T):
            hit += len(set(ex[l - 1, t].tolist()) & set(ex[l, t].tolist()))
    return hit / ((L - 1) * T * K)


TRACES = {'strong': make_trace(CFG['rho_strong'], 101),
          'weak': make_trace(CFG['rho_weak'], 202)}
for _g, _tr in TRACES.items():
    _tr['group'] = _g

print('%-8s %-10s %s' % ('组', 'rho', '相邻层同 token 专家重合率（随机基线 = %.3f）' % (K / E)))
for g, tr in TRACES.items():
    print('%-8s %-10.2f %.3f' % (g, tr['rho'], same_token_overlap(tr)))
print()
print('→ 强相关组重合率约是弱相关组的数倍：这就是"能预测"的物理来源。')

## 📐 指标定义（后面所有数字都由代码算出）

先把口径钉死，避免"命中率"一词被混用。所有指标都在**同一轨迹、同一预算**下比较。

| 指标 | 公式 | 含义 |
|---|---|---|
| 预取集合 $\mathcal{P}$ | 每波次输出 $m$ 个专家（`pf_width=4`） | 预算对齐：三种预取器等量输出 |
| 真实需求 $\mathcal{N}$ | 该波次内 token 实际激活的专家并集 | 按 microbatch 去重（DMA 以专家为单位） |
| 命中数 | $\lvert \mathcal{P}\cap\mathcal{N}\rvert$ | 预取对了的专家数 |
| `precision` | $\dfrac{\lvert \mathcal{P}\cap\mathcal{N}\rvert}{\lvert\mathcal{P}\rvert}$ | 预取体积里有多少是真需要的 |
| `recall`（命中率） | $\dfrac{\lvert \mathcal{P}\cap\mathcal{N}\rvert}{\lvert\mathcal{N}\rvert}$ | 真实需求有多少被提前搬到了 |
| **浪费率** `waste` | $1-\text{precision}=\dfrac{\lvert\mathcal{P}\setminus\mathcal{N}\rvert}{\lvert\mathcal{P}\rvert}$ | 预取了但没用到 |
| 基线 stall | $\text{stall}_{base}=\sum_{\text{波次}}\lvert\mathcal{N}\rvert\cdot t_{load}$ | 假设**完全无缓存**，每次都要装载 |
| 实际 stall | $\text{stall}_{pf}=\sum_{\text{波次}}\lvert\mathcal{N}\setminus\text{cache}\rvert\cdot t_{load}$ | 缓存未命中才等 |
| 预取开销 | $\text{overhead}=\sum_{\text{波次}}\lvert\mathcal{P}\rvert\cdot t_{dma}(1-\text{overlap})$ | 未被重叠掩盖的 DMA 成本 |
| **有效加速比** | $S=\dfrac{\text{stall}_{base}}{\text{stall}_{pf}+\text{overhead}}$ | $>1$ 才算真的更快 |

其中单专家 DMA 时延 $t_{dma}=\dfrac{\text{expert\_bytes}}{\text{BW}}=\dfrac{8\ \text{MB}}{48\ \text{GB/s}}\approx 167\ \mu s$，只有 $1-\text{overlap}=15\%$ 落在关键路径上。

**为什么 $S$ 的分母要加 overhead？** 因为预取不是免费搬运：即使完全重叠，也会抢 DMA 带宽、挤占缓存。只看 stall 不看 overhead，就会得出"预取得越多越好"的错误结论。

In [ ]:
def need_mask(tr):
    '''(L,C,E) 布尔：该波次真实需要哪些专家'''
    ex = tr['experts'].reshape(L, C, G, K)
    nm = np.zeros((L, C, E), dtype=bool)
    li = np.broadcast_to(np.arange(L)[:, None, None, None], ex.shape)
    ci = np.broadcast_to(np.arange(C)[None, :, None, None], ex.shape)
    nm[li, ci, ex] = True
    return nm


def markov_matrix(tr):
    '''从训练段统计转移矩阵 P(e_next | e_cur)，加 0.5 平滑（拉普拉斯）'''
    M = np.zeros((E, E), dtype=np.float64)
    ex = tr['experts'][:, :NTRAIN]
    for l in range(1, L):
        np.add.at(M, (ex[l - 1].reshape(-1), ex[l].reshape(-1)), 1.0)
    M += 0.5
    return M / M.sum(axis=1, keepdims=True)


for tr in TRACES.values():
    tr['nm'] = need_mask(tr)
    tr['P'] = markov_matrix(tr)
    freq = np.bincount(tr['experts'][:, :NTRAIN].reshape(-1), minlength=E).astype(np.float32)
    tr['freq_vec'] = np.argsort(-freq).astype(np.float32)      # 越小越热

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (g, tr) in zip(axes, TRACES.items()):
    # 只画训练段里出现频率最高的 12 个专家的子转移矩阵，看得清结构
    hot = np.argsort(-np.bincount(tr['experts'][:, :NTRAIN].reshape(-1), minlength=E))[:12]
    sub = tr['P'][np.ix_(hot, hot)]
    im = ax.imshow(sub, cmap='YlGnBu', vmin=0)
    ax.set_xticks(range(len(hot)), hot, fontsize=8)
    ax.set_yticks(range(len(hot)), hot, fontsize=8)
    ax.set_xlabel('下一层专家 $e_{next}$')
    ax.set_ylabel('当前层专家 $e_{cur}$')
    ax.set_title('转移矩阵（rho=%.2f，%s）' % (tr['rho'], '强相关' if g == 'strong' else '弱相关'))
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('马尔可夫预取器的统计源：对角线越亮 = 越容易预测', fontsize=12)
plt.tight_layout()

diag_s = np.diag(TRACES['strong']['P']).mean()
diag_w = np.diag(TRACES['weak']['P']).mean()
print('对角平均质量 P(e_next=e_cur)：强相关 %.3f vs 弱相关 %.3f' % (diag_s, diag_w))

plt.show()

## 4️⃣ 手写实现：三种预取器

统一接口（都只吃**发出时刻可得**的信息）：

```python
scores(tr, k) -> (L, C, E) 的分数张量     # 第 l 层用第 l-k 层的信息打分；k=0 表示不预取
```

**预算对齐**：三种预取器每波次都输出 $m=4$ 个专家（`pf_width`），否则 precision 不可比。
"无预取 / 纯 LRU / 全局热度"三种对照组也走同一套模拟器。

| 预取器 | 用到的信息 | 训练方式 | 预期 |
|---|---|---|---|
| `PrevLayerPrefetcher` | 第 $l-k$ 层的**路由 logits** top-m | 不需训练（层间相关基线） | 强相关时强，弱相关时崩 |
| `MarkovPrefetcher` | 第 $l-k$ 层**激活的专家** + 转移矩阵 $\mathbf{P}^{k}$ | 轨迹统计（闭式） | 比上层稳，但无 hidden 信息 |
| `LogisticPrefetcher` | hidden-state 投影 + 上一（$l-k$）层专家 one-hot + 层 one-hot | 手写 SGD + 早停 | 强相关时最好 |

> 为什么 Markov 要用 $\mathbf{P}^{k}$ 而不是 $\mathbf{P}$？因为预测第 $l$ 层时只能看到第 $l-k$ 层，中间隔了 $k$ 步转移，链式相乘 $\mathbf{P}^{k}$ 才对齐。

In [ ]:
def scores_prev(tr, k):
    '''上层预取：直接把第 l-k 层的路由分数 top-m 当作第 l 层的预测。

    这是最朴素的基线：不做任何统计，只赌"相邻层路由相似"。
    '''
    sc = np.zeros((L, C, E), dtype=np.float32)
    if k <= 0:
        return sc                                   # k=0：没有历史信息，不预取
    for l in range(k, L):
        sc[l] = tr['logits'][l - k].reshape(C, G, E).sum(axis=1)   # 按 microbatch 汇总
    return sc


def scores_freq(freq_vec, k):
    '''全局热度基线：不管 token，永远预取训练集里最热的 m 个专家'''
    sc = np.zeros((L, C, E), dtype=np.float32)
    if k > 0:
        sc[k:] = freq_vec
    return sc


tr = TRACES['strong']
sc = scores_prev(tr, 1)
l, c = 2, 0
need = np.flatnonzero(tr['nm'][l, c]).tolist()
pred = np.argsort(-sc[l, c])[:CFG['pf_width']].tolist()
print('示例：强相关组，第 %d 层、第 %d 个 microbatch' % (l, c))
print('  真实需要的专家 :', need)
print('  上层预取预测的 :', pred)
print('  命中           :', sorted(set(pred) & set(need)))
print('  该波次 precision=%.2f  recall=%.2f'
      % (len(set(pred) & set(need)) / len(pred), len(set(pred) & set(need)) / len(need)))

In [ ]:
def scores_markov(tr, k, conf=0.0):
    '''马尔可夫预取：把第 l-k 层的激活专家集合当作"权重"，用 P^k 扩散到第 l 层。

    rows[c, a] = 该 microbatch 里专家 a 被激活的次数；
    预测分布 = 归一化 rows @ P^k。
    conf 是置信阈值：低于阈值的候选直接丢掉（省带宽，但会掉 recall）。
    '''
    sc = np.zeros((L, C, E), dtype=np.float32)
    if k <= 0:
        return sc
    Pk = np.linalg.matrix_power(tr['P'], k)          # k 步链式转移
    for l in range(k, L):
        rows = np.zeros((C, E), dtype=np.float32)
        e = tr['experts'][l - k].reshape(C, G, K)
        for j in range(K):
            np.add.at(rows, (np.arange(C)[:, None], e[:, :, j]), 1.0)
        s = rows @ Pk
        if conf > 0:
            s[:, s.max(axis=1, keepdims=True) * conf > s] = 0.0
        sc[l] = s
    return sc


tr = TRACES['strong']
sm = scores_markov(tr, 1)
sp = scores_prev(tr, 1)
l, c = 3, 5
need = set(np.flatnonzero(tr['nm'][l, c]).tolist())
top_mk = np.argsort(-sm[l, c])[:CFG['pf_width']].tolist()
top_pv = np.argsort(-sp[l, c])[:CFG['pf_width']].tolist()
print('同一波次（strong, layer=%d, chunk=%d）两种预取器对比：' % (l, c))
print('  真实需求     :', sorted(need))
print('  上层预取 top4:', top_pv, '命中', sorted(set(top_pv) & need))
print('  马尔可夫 top4:', top_mk, '命中', sorted(set(top_mk) & need))
print('对角质量 P[e|e] 平均 = %.3f（越接近 1，马尔可夫越接近「复制上一层」）'
      % np.diag(tr['P']).mean())

In [ ]:
class LogisticPrefetcher:
    '''numpy 手写 multi-label logistic 回归（每个专家一个 logistic 单元）+ Adam + 早停。

    特征只用"发出时刻可得"的量：
      - hidden(l-k) 的低秩随机投影（捕捉语义/路由方向）
      - 第 l-k 层激活专家的 one-hot（捕捉「当前在哪」）
      - 那两个专家的路由 logit 强度（区分「险胜」与「碾压」）
      - 层 one-hot（捕捉「路由分布随层漂移」）
    目标：第 l 层该 token 的 top-k 专家（multi-label，用独立 sigmoid + BCE）。
    '''

    def __init__(self, cfg, seed=11):
        self.cfg, self.seed = cfg, seed

    def _feat(self, tr, k, rows):
        ti, li = rows % T, rows // T
        src = np.maximum(0, li - k)                       # 因果：只用 l-k 层
        proj = np.einsum('nf,hf->nh', tr['hid'][src, ti], self.P)
        ar = np.arange(len(rows))
        oh = np.zeros((len(rows), E), dtype=np.float32)
        oh[ar, tr['experts'][src, ti, 0]] = 1.0
        oh[ar, tr['experts'][src, ti, 1]] = 1.0
        lay = np.zeros((len(rows), L), dtype=np.float32)
        lay[ar, li] = 1.0
        # 发出层这两个专家的路由 logit 强度：区分「险胜」与「碾压」
        lg = tr['logits'][src, ti]
        lgt = np.stack([lg[ar, tr['experts'][src, ti, 0]], lg[ar, tr['experts'][src, ti, 1]]], axis=1)
        return np.concatenate([proj, oh, lgt, lay], axis=1)

    def fit(self, tr, k, verbose=False):
        c = self.cfg
        rng = np.random.default_rng(self.seed)
        self.P = rng.normal(0, 1 / np.sqrt(c['hidden']), (c['proj'], c['hidden'])).astype(np.float32)

        rows = np.concatenate([np.arange(l * T, l * T + NTRAIN) for l in range(max(1, k), L)])
        X = self._feat(tr, k, rows)
        Y = np.zeros((len(rows), E), dtype=np.float32)
        ei = tr['experts'].reshape(-1, K)
        ar = np.arange(len(rows))
        Y[ar, ei[rows, 0]] = 1.0
        Y[ar, ei[rows, 1]] = 1.0

        self.mu, self.sd = X.mean(axis=0), X.std(axis=0) + 1e-6
        X = (X - self.mu) / self.sd
        X = np.hstack([X, np.ones((len(rows), 1), dtype=np.float32)])

        idx = rng.permutation(len(rows))
        cut = int(0.72 * len(rows))                        # train / val 划分
        tri, vai = idx[:cut], idx[cut:]

        W = np.zeros((X.shape[1], E), dtype=np.float32)
        m_adam = np.zeros_like(W)
        v_adam = np.zeros_like(W)
        step = 0
        best = (np.inf, W.copy(), 0)
        hist = {'train': [], 'val': []}
        eps = 1e-7

        def bce(sel):
            p = 1.0 / (1.0 + np.exp(-(X[sel] @ W)))
            return float(-(Y[sel] * np.log(p + eps) + (1 - Y[sel]) * np.log(1 - p + eps)).mean())

        for ep in range(c['epochs']):
            order = rng.permutation(tri)
            for i in range(0, len(order), c['bs']):
                b = order[i:i + c['bs']]
                p = 1.0 / (1.0 + np.exp(-(X[b] @ W)))
                g = X[b].T @ (p - Y[b]) / len(b) + c['l2'] * W
                step += 1
                m_adam = 0.9 * m_adam + 0.1 * g
                v_adam = 0.999 * v_adam + 0.001 * g * g
                W -= c['lr'] * (m_adam / (1 - 0.9 ** step)) / (np.sqrt(v_adam / (1 - 0.999 ** step)) + 1e-8)
            hist['train'].append(bce(tri))
            hist['val'].append(bce(vai))
            if hist['val'][-1] < best[0] - 1e-7:            # 早停：只保留 val 最优
                best = (hist['val'][-1], W.copy(), ep)
            elif ep - best[2] > c['patience']:
                break

        self.W, self.hist, self.best_ep, self.best_val = best[1], hist, best[2], best[0]
        if verbose:
            print('    k=%d: 训练到第 %d 轮早停，最优 val BCE=%.4f' % (k, len(hist['val']), self.best_val))
        return self

    def scores(self, tr, k):
        rows = np.arange(L * T)
        X = self._feat(tr, k, rows)
        X = (X - self.mu) / self.sd
        X = np.hstack([X, np.ones((len(rows), 1), dtype=np.float32)])
        S = 1.0 / (1.0 + np.exp(-(X @ self.W)))
        sc = S.reshape(L, C, G, E).sum(axis=2)             # 按 microbatch 汇总
        sc[:max(1, k)] = 0.0        # 因果：第 0 层（以及 k 之前的层）没有上一层的可用信息
        return sc


LOG = {}
print('训练 logistic 预取器（每组一个模型，按 k=1 的信息集训练）:')
for g in ('strong', 'weak'):
    LOG[g] = LogisticPrefetcher(CFG).fit(TRACES[g], 1, verbose=True)
print('累计 %.1fs' % (time.time() - t_start))
print()
print('工程取舍：预测器只在 k=1 上训练；k>=2 时不重训，直接复用同一模型去预测更远的层。')
print('这样既省下训练成本，也真实反映了"固定预测器看更远景"的分布偏移惩罚。')

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 3.9))
for g, col in (('strong', '#dc2626'), ('weak', '#2563eb')):
    h = LOG[g].hist
    ep = np.arange(len(h['val']))
    ax.plot(ep, h['train'], lw=1.5, color=col, label='%s 训练损失' % g)
    ax.plot(ep, h['val'], lw=1.9, ls='--', color=col, label='%s 验证损失' % g)
    best_ep = LOG[g].best_ep
    ax.axvline(best_ep, color=col, lw=0.8, alpha=0.3)
    ax.scatter([best_ep], [h['val'][best_ep]], s=40, color=col, zorder=5)
ax.set_xlabel('epoch')
ax.set_ylabel('BCE 损失（每专家二元交叉熵）')
ax.set_title('logistic 预取器训练曲线（虚竖线 = 早停选中的轮次）')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()

for g in ('strong', 'weak'):
    m = LOG[g]
    print('%-6s 最优 epoch=%d，train=%.4f，val=%.4f，共跑 %d 轮'
          % (g, m.best_ep, m.hist['train'][m.best_ep], m.best_val, len(m.hist['val'])))
print('\n观察：弱相关组的验证损失明显更高 —— hidden-state 里几乎没有关于下一层路由的信号，')
print('      这正是它在弱相关组只能退化为"条件先验估计"的原因。')

plt.show()

## 🔧 组合模拟：专家缓存 + 波次执行

预取器只决定"搬什么"，真正决定快慢的是**缓存与 DMA 的交互**。模拟器按真实执行顺序跑：

```
for l in layers:                       # 层优先（MoE 逐层执行）
    for c in chunks:                   # 每个 microbatch 一个波次
        1. 预取数据到达 → 插入 LRU 缓存（占槽位，可能挤掉热专家）
        2. 按需访问真实需求集合 → 未命中的算 stall
        3. 刷新 LRU（命中的移到队尾）
```

两个关键设计：

- **DMA 单位是专家，不是 token**：一个波次里 8 个 token 可能摸到 10 多个专家，所以用并集 $\mathcal{N}$ 去重；
- **预取会污染缓存**：这是 wasteful prefetch 的真正伤害 —— 它不仅浪费带宽，还把本来能命中的热专家挤出去。

对照组的语义：

| 策略 | 缓存 | 预取 | 说明 |
|---|---|---|---|
| 无预取 | ✗ | ✗ | 每次都从外存装载，$S$ 恒为 1（基准） |
| 纯 LRU | ✓ | ✗ | 只靠缓存复用，不预测 |
| 全局热度 | ✓ | ✓ | 静态预取最热的 $m$ 个（不看 token） |
| 上层预取 | ✓ | ✓ | 复用上一层路由 top-m |
| 马尔可夫 | ✓ | ✓ | $\mathbf{P}^{k}$ 扩散 |
| logistic | ✓ | ✓ | hidden-state + 历史专家特征 |

In [ ]:
class ExpertCache:
    '''dict 实现的 O(1) LRU：插入顺序即 LRU 顺序，pop 最早插入的即淘汰'''

    def __init__(self, slots):
        self.slots = int(slots)
        self.d = {}

    def touch(self, e):
        self.d.pop(e, None)
        self.d[e] = 1
        if len(self.d) > self.slots:
            self.d.pop(next(iter(self.d)))

    def __contains__(self, e):
        return e in self.d


def simulate(name, k, sc, nm, cfg, slots=None, use_cache=True):
    '''跑完整个轨迹，返回 precision/recall/waste/stall/overhead/speedup'''
    slots = cfg['cache_slots'] if slots is None else slots
    pf_cost = cfg['expert_mb'] * 1000.0 / cfg['dma_gbs'] * (1.0 - cfg['overlap'])

    pf = np.zeros_like(nm)
    pf_active = (name not in ('无预取', '纯LRU')) and (sc is not None) and (k > 0)
    if pf_active:
        idx = np.argsort(-sc, axis=-1)[..., :cfg['pf_width']]
        np.put_along_axis(pf, idx, True, axis=-1)
        pf &= (sc.max(axis=-1) > 0)[..., None]      # 分数全零 = 该层没有可用信息，不预取

    need_i = [[np.flatnonzero(nm[l, c]).tolist() for c in range(C)] for l in range(L)]
    pf_i = [[np.flatnonzero(pf[l, c]).tolist() for c in range(C)] for l in range(L)]

    cache = ExpertCache(slots)
    stall = overhead = 0.0
    n_need = n_pf = n_hit = n_miss = 0

    for l in range(L):
        for c in range(C):
            nd, p = need_i[l][c], pf_i[l][c]
            n_need += len(nd)
            n_pf += len(p)
            n_hit += len(set(nd) & set(p))
            if use_cache:
                for e in p:                       # 1) 预取到达，先占槽位（可能挤掉热专家）
                    cache.touch(e)
                mis = [e for e in nd if e not in cache]
                for e in nd:                      # 2) 真正的按需访问
                    cache.touch(e)
            else:
                mis = nd
            n_miss += len(mis)
            stall += len(mis) * cfg['load_us']
            overhead += len(p) * pf_cost

    base = n_need * cfg['load_us']
    prec = n_hit / n_pf if n_pf else 0.0
    return dict(strategy=name, k=k, precision=prec, recall=n_hit / n_need,
                waste=(1.0 - prec) if n_pf else 0.0,
                stall_ms=stall / 1000.0, overhead_ms=overhead / 1000.0,
                speedup=base / (stall + overhead), miss=n_miss / n_need,
                n_hit=n_hit, n_pf=n_pf)


def scores_for(tr, name, k):
    if name == '全局热度':
        return scores_freq(tr['freq_vec'], k)
    if name == '上层预取':
        return scores_prev(tr, k)
    if name == '马尔可夫':
        return scores_markov(tr, k)
    if name == 'logistic':
        return LOG[tr['group']].scores(tr, k)      # 同一模型用于任意提前层数 k
    return None


_SCORE_CACHE = {}


def get_scores(tr, name, k):
    '''分数张量缓存：logistic 打分是整条流水线里唯一较贵的纯计算，只算一次'''
    key = (tr['group'], name, k)
    if key not in _SCORE_CACHE:
        _SCORE_CACHE[key] = scores_for(tr, name, k)
    return _SCORE_CACHE[key]


STRATS = ['无预取', '纯LRU', '全局热度', '上层预取', '马尔可夫', 'logistic']
demo = simulate('logistic', 1, get_scores(TRACES['strong'], 'logistic', 1), TRACES['strong']['nm'], CFG)
print('单次模拟自检（strong, logistic, k=1）：')
for kk, vv in demo.items():
    print('  %-12s %s' % (kk, ('%.4f' % vv) if isinstance(vv, float) else vv))

## 5️⃣ 数值实验：命中率 vs 浪费率 vs 有效加速比

四组实验，全部由代码算出：

1. **六策略对比**（强/弱两组）：`precision / recall / waste / miss / S` —— 看谁真的更快；
2. **提前层数扫描** $k\in\{0,1,2,3\}$：命中率-浪费率权衡曲线 —— 看得越远越不划算；
3. **缓存槽位敏感性**：缓存越小，预取的边际收益越大（还是越小？）；
4. **逐层浪费率分解**：浪费发生在哪些层。

先看六策略主表。

In [ ]:
RES = {}
print('%-8s %-10s %8s %8s %8s %8s %8s %9s' %
      ('组', '策略', 'prec', 'recall', 'waste', 'miss', 'S', 'stall(ms)'))
print('-' * 84)
for g, tr in TRACES.items():
    for s in STRATS:
        r = simulate(s, 1, get_scores(tr, s, 1), tr['nm'], CFG, use_cache=(s != '无预取'))
        RES[(g, s, 1)] = r
        print('%-8s %-10s %8.3f %8.3f %8.3f %8.3f %8.2f %9.1f'
              % (g, s, r['precision'], r['recall'], r['waste'], r['miss'], r['speedup'], r['stall_ms']))
    print('-' * 84)

best = max((r for key, r in RES.items()), key=lambda r: r['speedup'])
print('有效加速比最高：%s（S=%.2f）。注意 phantom 收益：' % (best['strategy'], best['speedup']))
print('  纯 LRU 的 S=%.2f 说明"什么都不预测、只靠缓存复用"已经吃掉大部分收益；'
      % RES[('strong', '纯LRU', 1)]['speedup'])
print('  预取要在它之上再加分，必须命中率高到能抵掉 DMA overhead。')

### 断言 ①②④：预取器能力与端到端收益

把上面这张表的关键关系钉成断言（用区间，不用魔法常数）：

- **① 强相关组的上层预取 recall 明显高于弱相关组**：层间相关是"能预测"的物理前提；
- **② 同一轨迹同一 $k$ 下 logistic 的 precision 不低于马尔可夫**：强相关组显著领先（hidden-state 多带一份信息），弱相关组两者退化到同一「条件先验」水平、落在统计等价区间；
- **④ 至少存在一个策略的有效加速比 > 1**：否则"预取"就是纯亏本买卖。

In [ ]:
prev_s = RES[('strong', '上层预取', 1)]['recall']
prev_w = RES[('weak', '上层预取', 1)]['recall']
lg_s = RES[('strong', 'logistic', 1)]['precision']
mk_s = RES[('strong', '马尔可夫', 1)]['precision']
lg_w = RES[('weak', 'logistic', 1)]['precision']
mk_w = RES[('weak', '马尔可夫', 1)]['precision']

print('① 上层预取 recall：strong=%.3f  weak=%.3f  比值=%.2f×' % (prev_s, prev_w, prev_s / max(prev_w, 1e-9)))
print('② logistic vs 马尔可夫 precision：')
print('     strong: %.3f vs %.3f  (+%.3f)' % (lg_s, mk_s, lg_s - mk_s))
print('     weak  : %.3f vs %.3f  (%+.3f)' % (lg_w, mk_w, lg_w - mk_w))

# ① 强相关组必须明显更高：绝对差 > 0.03 且比值 > 1.5
assert prev_s >= prev_w + 0.03, '强相关组上层预取 recall 未显著高于弱相关组'
assert prev_s / max(prev_w, 1e-9) >= 1.5, '强弱两组的上层预取 recall 比值不够大'

# ② 强相关组：logistic 显著领先（hidden-state 特征真的多带了一份信息）；
#    弱相关组：hidden 不含下一层路由信号，两者都退化成「给定当前专家集合的条件先验估计」，
#    差异只剩余估计方差 → 断言写成区间等价（|Δ| <= 0.02），并要求跨组平均仍领先。
assert lg_s >= mk_s + 0.02, '强相关组 logistic 未显著优于马尔可夫'
assert abs(lg_w - mk_w) <= 0.02, '弱相关组 logistic 与马尔可夫超出统计等价区间'
assert (lg_s + lg_w) / 2 > (mk_s + mk_w) / 2, 'logistic 跨组平均精度未超过马尔可夫'

# ④ 至少一个策略真的更快
assert max(r['speedup'] for r in RES.values()) > 1.05, '没有任何策略实现有效加速'
print('\n断言 ①②④ 全部通过')

In [ ]:
# 提前层数扫描：k=0 表示"不预取"（等价纯 LRU 基线）
SWEEP = {}
KS = [0, 1, 2, 3]
for g, tr in TRACES.items():
    zero = simulate('上层预取', 0, None, tr['nm'], CFG)     # k=0：不预取，与策略无关
    for s in ['上层预取', '马尔可夫', 'logistic']:
        SWEEP[(g, s, 0)] = dict(zero, strategy=s)
        for k in KS[1:]:
            SWEEP[(g, s, k)] = simulate(s, k, get_scores(tr, s, k), tr['nm'], CFG)

print('%-8s %-10s %5s %8s %8s %8s %7s' % ('组', '策略', 'k', 'recall', 'waste', 'prec', 'S'))
print('-' * 58)
for g in TRACES:
    for s in ['上层预取', '马尔可夫', 'logistic']:
        for k in KS:
            r = SWEEP.get((g, s, k))
            if r is None:
                continue
            print('%-8s %-10s %5d %8.3f %8.3f %8.3f %7.2f'
                  % (g, s, k, r['recall'], r['waste'], r['precision'], r['speedup']))
    print('-' * 58)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
styles = {'上层预取': ('#2563eb', 'o', (7, 6)), '马尔可夫': ('#16a34a', 's', (-26, 9)),
          'logistic': ('#dc2626', '^', (7, -14))}
for ax, g in zip(axes, ['strong', 'weak']):
    for s in ['上层预取', '马尔可夫', 'logistic']:
        pts = [(SWEEP[(g, s, k)]['recall'], SWEEP[(g, s, k)]['waste'], k)
               for k in KS if (g, s, k) in SWEEP]
        if not pts:
            continue
        col, mk, off = styles[s]
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        ax.plot(xs, ys, marker=mk, color=col, lw=1.8, label=s)
        for x, y, k in pts[1:]:                      # k=0 是原点，不标
            ax.annotate('k=%d' % k, (x, y), textcoords='offset points', xytext=off,
                        fontsize=8, color=col,
                        bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.8))
    ax.margins(x=0.12, y=0.14)
    ax.set_xlabel('命中率 recall（越高越好 →）')
    ax.set_ylabel('浪费率 waste（越低越好 ↓）')
    ax.set_title('%s组：命中率-浪费率权衡（数字 = 提前层数 k）原点 = 不预取' % ('强相关' if g == 'strong' else '弱相关'))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9, loc='lower right')
plt.suptitle('预测得越远（k 越大）：命中率下降、浪费率上升 —— 这就是预取的真实代价', fontsize=12)
plt.tight_layout()

print('读图：')
print('  · 曲线左下角 = 又准又省；右上角 = 又烂又贵。k 增大时点一路向右上漂。')
print('  · 强相关组整体更靠左下（可预测性强），弱相关组几乎贴着"高浪费"边缘。')

plt.show()

### 断言 ③：k 增大 ⟹ 浪费率单调不减

这条是"预测越远越不靠谱"的定量版本。用上层预取与马尔可夫的均值检验（两者在全部 $k$ 上都有定义）：

$$
\overline{\text{waste}}(k)\ \text{关于}\ k\ \text{单调不减}.
$$

$k=0$ 定义为"不预取"，此时 $\lvert\mathcal{P}\rvert=0$，浪费率按定义为 0。

In [ ]:
for g in ['strong', 'weak']:
    means = []
    for k in KS:
        vals = [SWEEP[(g, s, k)]['waste'] for s in ['上层预取', '马尔可夫'] if (g, s, k) in SWEEP]
        means.append(float(np.mean(vals)))
    diffs = np.diff(means)
    print('%-6s 浪费率均值 by k: %s' % (g, ' → '.join('%.3f' % v for v in means)))
    print('       相邻差分      : %s' % np.array2string(diffs, precision=3))
    # 区间断言：允许 2e-3 的有限样本噪声（弱相关组在 k>=2 已饱和，差分只剩千分位抖动）
    assert np.all(diffs >= -2e-3), '%s 组浪费率随 k 并非单调不减' % g

    # 交叉验证：k>=1 之后命中率应单调不增（k=0 不预取，命中率按定义为 0，不参与）
    rec = [float(np.mean([SWEEP[(g, s, k)]['recall'] for s in ['上层预取', '马尔可夫']
                          if (g, s, k) in SWEEP])) for k in KS if k >= 1]
    assert np.all(np.diff(rec) <= 1e-9), '%s 组命中率随 k 并非单调不增' % g
    print('       命中率均值(k>=1): %s（单调不增 ✓）' % ' → '.join('%.3f' % v for v in rec))
print('\n断言 ③ 通过：k 越大，浪费率单调越高、命中率单调越低。')
print('工程含义：预取最多提前 1~2 层；提前太久，省下的 stall 抵不过多搬的无效权重。')

In [ ]:
# 缓存槽位敏感性：同样预测器，缓存越小，"预取挤占缓存"的副作用越明显
SLOT_GRID = [4, 16, 48]
slot_res = {}
for g, tr in TRACES.items():
    for s in ['纯LRU', 'logistic']:
        sc = get_scores(tr, s, 1)                       # 分数与槽位无关，复用
        slot_res[(g, s)] = [(sl, simulate(s, 1, sc, tr['nm'], CFG, slots=sl)) for sl in SLOT_GRID]

SLOW = {(g, s): [] for g in TRACES for s in ['logistic']}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))
for ax, g in zip(axes, ['strong', 'weak']):
    base = [r['speedup'] for _, r in slot_res[(g, '纯LRU')]]
    ax.plot(SLOT_GRID, base, marker='o', lw=2.2, ls='--', color='#475569', label='纯 LRU（不预取）：S')
    for s, col in (('logistic', '#dc2626'),):
        sv = [r['speedup'] for _, r in slot_res[(g, s)]]
        ax.plot(SLOT_GRID, sv, marker='s', lw=1.8, color=col, label='%s：S' % s)
        gap = np.array(sv) - np.array(base)
        for sl_idx in range(len(SLOT_GRID)):
            if gap[sl_idx] < 0:                     # 该槽位下预取比纯 LRU 更慢
                ax.scatter([SLOT_GRID[sl_idx]], [sv[sl_idx]], s=150, marker='v',
                           facecolors='none', edgecolors=col, lw=1.4, zorder=6)
                SLOW[(g, s)].append(SLOT_GRID[sl_idx])
    ax.plot([], [], marker='v', ls='none', markerfacecolor='none', markeredgecolor='#475569',
            markeredgewidth=1.4, label='空心三角 = 预取比纯 LRU 慢')
    ax.set_xlabel('缓存槽位数（×8MB；全量 64 个专家 = 64 槽）')
    ax.set_ylabel('有效加速比 S')
    ax.set_title('%s组：缓存越大，预取的边际价值越小甚至转负' % ('强相关' if g == 'strong' else '弱相关'))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()

print('空心倒三角标记的含义：该槽位下 "cache + 预取" 的有效加速比低于 "cache only"，')
print('也就是预取只在白烧带宽、还挤掉了本该命中的热专家。')
for (g, s), slots in SLOW.items():
    print('  %-6s %-9s 预取为负收益的槽位数：%s' % (g, s, slots if slots else '无'))

print('缓存 48 槽（≈ 全部 64 专家的 75%）时：')
for g in TRACES:
    lru = slot_res[(g, '纯LRU')][-1][1]['speedup']
    lg = slot_res[(g, 'logistic')][-1][1]['speedup']
    print('  %-6s 纯LRU S=%.2f vs logistic S=%.2f（差 %+.2f）—— 大部分专家常驻后，无用预取只是在挤缓存'
          % (g, lru, lg, lg - lru))
print('缓存 4 槽（极紧）时：')
for g in TRACES:
    lru = slot_res[(g, '纯LRU')][0][1]['speedup']
    lg = slot_res[(g, 'logistic')][0][1]['speedup']
    print('  %-6s 纯LRU S=%.2f vs logistic S=%.2f（差 %+.2f）—— 缓存太紧时预取挤占严重，收益被吃掉'
          % (g, lru, lg, lg - lru))

plt.show()

In [ ]:
# 逐层浪费率分解：浪费集中在哪些层？
def per_layer_waste(tr, s, k=1, sc=None):
    '''逐层浪费率；没有可用预测的层（第 0 层、k 之前）返回 nan'''
    sc = get_scores(tr, s, k) if sc is None else sc
    pf = np.zeros_like(tr['nm'])
    np.put_along_axis(pf, np.argsort(-sc, axis=-1)[..., :CFG['pf_width']], True, axis=-1)
    pf &= (sc.max(axis=-1) > 0)[..., None]        # 与 simulate 一致：零分数 = 不预取
    out = []
    for l in range(L):
        p = pf[l].sum()
        hit = (pf[l] & tr['nm'][l]).sum()
        out.append(1.0 - hit / p if p else np.nan)
    return np.array(out)

fig, ax = plt.subplots(figsize=(7.6, 3.9))
xs = np.arange(L)
for s, col in (('上层预取', '#2563eb'), ('马尔可夫', '#16a34a'), ('logistic', '#dc2626')):
    ax.plot(xs, per_layer_waste(TRACES['strong'], s), marker='o', color=col, lw=1.8, label='strong · %s' % s)
    ax.plot(xs, per_layer_waste(TRACES['weak'], s), marker='x', color=col, lw=1.2, ls='--',
            label='weak · %s' % s)
ax.set_xlabel('层号 l（预取对象层）')
ax.set_ylabel('该层浪费率')
ax.set_xticks(xs)
ax.set_title('逐层浪费率：第 0 层没有上一层信息，无法预取（曲线从第 1 层开始）')
ax.grid(alpha=0.3)
ax.legend(fontsize=7.5, ncol=2)
plt.tight_layout()

w_mk = per_layer_waste(TRACES['strong'], '马尔可夫')
print('马尔可夫逐层浪费率：', np.array2string(w_mk, precision=3), '（nan = 该层无可用预测）')
print('第 0 层没有上一层信息 → 不预取（浪费率记为 nan）；第 1 层起稳定在 %.2f 附近。'
      % np.nanmean(w_mk[1:]))

plt.show()

In [ ]:
# 一个波次的执行时间线（定性理解 stall 与浪费如何叠加）
tr = TRACES['strong']
sc = get_scores(tr, 'logistic', 1)
pf = np.zeros_like(tr['nm'])
np.put_along_axis(pf, np.argsort(-sc, axis=-1)[..., :CFG['pf_width']], True, axis=-1)

l, c = 2, 3
need = np.flatnonzero(tr['nm'][l, c]).tolist()
pred = np.flatnonzero(pf[l, c]).tolist()
hit = sorted(set(need) & set(pred))
wasted = sorted(set(pred) - set(need))
miss = sorted(set(need) - set(pred))

print('时间线示例：强相关组，第 %d 层，第 %d 个 microbatch（%d 个 token）' % (l, c, G))
print('  ① router 输出（真实需求）: %s' % need)
print('  ② 预取预测(top-%d)        : %s' % (CFG['pf_width'], pred))
print('  ③ 命中（已在缓存，直接算）: %s  → 省下 %d × %.0f µs stall'
      % (hit, len(hit), CFG['load_us']))
print('  ④ 未命中（串行装载）      : %s  → 付出 %d × %.0f µs stall'
      % (miss, len(miss), CFG['load_us']))
print('  ⑤ 预取了但没用（浪费）    : %s  → 付出 %d × %.1f µs DMA 带宽'
      % (wasted, len(wasted), CFG['expert_mb'] * 1000.0 / CFG['dma_gbs'] * (1 - CFG['overlap'])))
print()
print('该波次：stall=%.0f µs，预取 overhead=%.1f µs，理论净收益=%.1f µs'
      % (len(miss) * CFG['load_us'],
         len(pred) * CFG['expert_mb'] * 1000.0 / CFG['dma_gbs'] * (1 - CFG['overlap']),
         len(hit) * CFG['load_us'] - len(pred) * CFG['expert_mb'] * 1000.0 / CFG['dma_gbs'] * (1 - CFG['overlap'])))

## 6️⃣ 工程对照

定性对照三个真实系统的设计取向（不做倍数/编号层面的编造，只对齐机制）：

| 系统 | 核心思路 | 与本节的关系 |
|---|---|---|
| **ExpertFlow** | 路由预测 + 分级缓存（GPU / CPU / NVMe），预测下一层要哪些专家并提前换入 | 就是本节"上层/马尔可夫/logistic 预测器 + LRU 缓存"的完整工程版；它同样要处理"predicted-but-unused"的浪费 |
| **HOBBIT** | 混合精度专家 + 预取：热专家留高精度、冷专家低精度/放远端，配合预取掩盖换入时延 | 提醒我们预取的收益和"专家体积"强耦合：专家越小，$t_{dma}$ 越小，预取的性价比越高 |
| **ProMoE** | 分组预取：把专家按共现关系分组，一次预取整组而非单个专家 | 对应本节的一个隐含假设——我们把每个专家当独立单位；真实系统里"整组搬"能摊薄 DMA 启动开销 |

三个共性结论：

1. **预测器不是越复杂越好**：上层预取就已经拿到相当一部分收益，复杂模型只在层间相关足够强时才划算（对应断言 ①）；
2. **预取必须和缓存策略一起设计**：预取会污染 LRU，热专家被挤掉是浪费率的隐性成本（对应槽位敏感性实验）；
3. **预取的收益随"专家/缓存比"变化**：缓存装得下大半专家时，预取几乎无事可做。

### 投机预取的代价：带宽竞争与置信阈值早停

预取本质是**投机执行**（speculative execution），它的三类代价：

| 代价 | 机制 | 本节对应指标 |
|---|---|---|
| 带宽占用 | DMA 抢走 HBM/PCIe 带宽，拖慢正常装载与计算 | `overhead_ms` |
| 缓存污染 | 无用专家挤掉热专家，反而制造新的 stall | 槽位敏感性曲线 |
| 功耗/尾延迟 | 无效搬运带来的额外功耗与长尾 | 未建模（定性） |

**置信阈值早停**（对应 `scores_markov(..., conf=)`）：只预取"足够有把握"的候选，宁可少预取也不乱预取。其效果是**用 recall 换 precision** —— 在浪费率敏感的部署里，这通常比"把 top-m 塞满"更划算。

**什么时候该关掉预取？**

- 层间相关弱（弱相关组）+ 缓存已经比较大 → $S$ 提升很小甚至为负；
- 专家很大（$t_{dma}$ 高）+ 批次很小（microbatch 需求集合小）→ overhead 摊不薄；
- 带宽本就紧张（其他 kernel 在抢）→ 预取会拉长关键路径。

> 面试金句：**预取不是"免费加速"，而是"用带宽买时延"的杠杆；杠杆率由命中率与浪费率共同决定。**

## 📊 指标口径速查（面试可直接背）

| 指标 | 公式 | 陷阱 |
|---|---|---|
| precision | $\lvert\mathcal{P}\cap\mathcal{N}\rvert/\lvert\mathcal{P}\rvert$ | 预取体积受预算 $m$ 影响，跨预算不可比 |
| recall（命中率） | $\lvert\mathcal{P}\cap\mathcal{N}\rvert/\lvert\mathcal{N}\rvert$ | $\mathcal{N}$ 是 microbatch 去重后的并集，不是 token 数 |
| waste | $1-\text{precision}$ | 只看 waste 不看 stall 会误判 |
| miss rate | $\lvert\mathcal{N}\setminus\text{cache}\rvert/\lvert\mathcal{N}\rvert$ | 与 recall 不同：缓存可能已经装着 |
| speedup | $S=\text{stall}_{base}/(\text{stall}_{pf}+\text{overhead})$ | 分母必须双向计入，否则永远"预取越多越好" |
| amortization | $t_{dma}(1-\text{overlap})$ | 重叠比例是实测参数，不是常数 |

一句话记忆：**precision/waste 描述"猜得准不准"，recall 描述"够不够用"，S 描述"到底值不值"。三者缺一都不能下结论。**

## 🎯 面试考点

1. **MoE 为什么需要专家换入/预取**：专家总量远超显存，单 token 只激活 top-k，激活集合数据相关；
2. **预测的信息来源**：上一层已算完的路由分数、激活专家、hidden-state；**因果边界**是不能用到目标层自身信息；
3. **三种预测器**：上层复用（基线）、马尔可夫转移（统计）、logistic/MLP（特征学习），各自适用条件；
4. **为什么马尔可夫要用 $\mathbf{P}^k$**：提前 $k$ 层时中间隔了 $k$ 步转移；
5. **浪费率为什么关键**：预取未经使用的权重仍占 DMA 带宽、污染缓存；
6. **有效加速比的正确口径**：分母同时含 stall 与 overhead，否则会高估预取收益；
7. **缓存与预取的耦合**：预取会挤掉热专家，LRU 与预取策略必须联合调优；
8. **兜底方案**：全局热度静态预取、置信阈值早停、按需降级；
9. **与相邻技术的关系**：专家分页（11 篇）、卸载流水（12 篇）、MoE 基础（04/16 篇）。

## 🎤 高频追问 Q&A

**Q1：为什么不把所有专家常驻显存？**
A：$E$ 个专家 × 每层 × 参数量的总量常常是显存的好几倍；而且 MoE 的稀疏性意味着大量专家在单个 batch 里根本不会被激活，常驻是纯浪费。

**Q2：为什么"上一层预取"这种土办法有效？**
A：因为残差流让相邻层的 hidden-state 高度相关，路由分数又由 hidden 决定，所以相邻层的激活专家高度重合。本节强相关组的重合率可以实测出来。

**Q3：预测错了会怎样？只是慢一点吗？**
A：不是。错误的预取会（a）占用 DMA 带宽、（b）把缓存里的热专家挤出去，制造**额外的** stall。所以浪费率必须单独度量。

**Q4：为什么不干脆预取全部专家？**
A：等价于"全量常驻"，带宽与时延都不可承受。预取的预算是硬约束（带宽 × 时间窗口）。

**Q5：加速比为什么不用"命中率"直接折算？**
A：命中率只说明猜对多少，不说明省下多少时延。真正的收益是 $\text{stall}$ 的减少量与 `overhead` 的对比，还受缓存基线影响（纯 LRU 已经吃掉大部分收益）。

**Q6：$\mathbf{P}^k$ 会不会数值退化？**
A：会。$k$ 大时 $\mathbf{P}^k$ 趋向平稳分布（各行趋同），预测退化为"全局热度"。这正是"k 越大浪费率越高"的另一面。

## ⚠️ 常见误区

| 误区 | 为什么错 |
|---|---|
| "预取越多越好" | 预取体积受带宽约束；无用预取挤占缓存，可能让 $S$ 变小 |
| "命中率高就一定快" | 还要看被省下的 stall 有多大、overhead 有多高；纯 LRU 基线可能已经很强 |
| "精度只看 top-1" | MoE 一次要 top-k 个专家，预取预算 $m$ 通常 > k，必须看集合级 precision/recall |
| "用目标层的路由结果做特征" | 因果泄漏：目标层路由要等该层 hidden 算完才有 |
| "Markov 一阶就够" | 提前 $k$ 层需要 $\mathbf{P}^k$；但 $k$ 大时矩阵幂会趋近平稳分布，收益衰减 |
| "预取和缓存独立调参" | 预取会污染 LRU，二者必须联合评估 |

## 📖 术语速查

| 术语 | 含义 |
|---|---|
| MoE | Mixture-of-Experts，稀疏激活的专家混合层 |
| top-k routing | 每 token 每层选 k 个专家 |
| expert offload / swap | 专家权重在 GPU 与 CPU/NVMe 之间换入换出 |
| prefetch | 提前把未来要用的权重搬入缓存 |
| speculative prefetch | 预测式（投机）预取，可能猜错 |
| precision@m / recall@m | 预取集合的准确率 / 覆盖率 |
| waste rate | 预取未被使用的比例 |
| stall | 因等待装载而暴露的空转时延 |
| LRU | 最近最少使用淘汰策略 |
| microbatch wave | 逐层执行时的一个调度波次（本文 = 8 token） |
| $\mathbf{P}^k$ | 转移矩阵的 k 次幂，用于 k 步预测 |

## ✅ 自测清单 + 衔接

- [ ] 能说清"为什么不常驻全部专家"，并能估算 $\text{stall}_{base}$ 的量级；
- [ ] 能手写上层预取基线，并解释它为什么在强相关组有效；
- [ ] 能从轨迹统计出 $\mathbf{P}$，并解释为什么提前 $k$ 层要用 $\mathbf{P}^k$；
- [ ] 能手写 logistic 预取器（特征、train/val、早停），并说明因果边界；
- [ ] 能写出 precision / recall / waste / $S$ 的公式，并指出 $S$ 分母为什么必须含 overhead；
- [ ] 能解释"预取污染缓存"如何让浪费率变成真实成本；
- [ ] 能复述"k 越大 → 命中率降、浪费率升"的原因与工程含义（最多提前 1~2 层）；
- [ ] 能说出 ExpertFlow / HOBBIT / ProMoE 各自的设计取向。

**衔接**：

- **11 篇《专家分页与显存池化》**：那里讲"显存装不下就分页"，本篇讲"分页之后怎么少等" —— 分页是存储侧，预取是时延侧；
- **12 篇《专家卸载与 CPU-GPU 协同》**：本篇的 `overlap=0.85` 就是那条卸载流水的抽象；真正的重叠要靠双缓冲 + 独立 stream；
- **04 号模块的 16 篇《MoE 混合专家》**：路由的稀疏性与负载均衡噪声，决定了本篇预测器的上限；
- 一句话总结：**预取的成败不取决于模型多复杂，而取决于"路由到底有多可预测"以及"猜错的代价有多大"。**

> 全部数字由上方代码实时算出（固定 seed，可复现）。